In [0]:
%pip install langchain==0.1.14 typing_extensions==4.8.0 --force-reinstall
dbutils.library.restartPython()


In [0]:
%pip install -U langchain langchain-community


In [0]:
%pip install mlflow[databricks] --upgrade
dbutils.library.restartPython()

In [0]:
import pandas as pd
import io
from langchain_community.chat_models import ChatDatabricks  # ✅ Correct module for LLM on Databricks

# Step 1: Load Unity Catalog table
df = spark.table("solutions_catalog.pih_schema.brand_data")

# Step 2: Convert to Pandas CSV
pdf = df.limit(10).toPandas()
csv_buffer = io.StringIO()
pdf.to_csv(csv_buffer, index=False)
csv_content = csv_buffer.getvalue()

# Step 3: Prepare Prompt
dataset_knowledge = """
This dataset contains brand campaign performance metrics.
- `kpi_metric_name`: KPI being measured
- `mapping`: channel grouping
- `control_pct`, `exposed_pct`: observed values
- `lift_pct`: percentage difference between control and exposed
"""

prompt = f"""
You are a campaign performance analyst.

Given this data:
{csv_content}

Instructions:
- Summarize which KPIs have the highest and lowest lift_pct.
- Identify any unusual behavior.
- Provide insights related to platform mapping or control/exposed group differences.
"""

# Step 4: Run Databricks-hosted LLM
llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", max_tokens=2048)
response = llm.invoke(prompt)

# Step 5: Display result
print("Insight from LLaMA 3.3:")
print(response.content.strip())


Problem	How RAG Solves It

Too much data to fit in prompt	- Only the most relevant rows are retrieved

Need contextual insights	- Retrieved rows provide grounding for the LLM

Questions vary by user intent	- LLM can reason flexibly with retrieved info

Slow full scan with LLM	- Vector DB allows fast retrieval + summaries